In [1]:
import pandas as pd

df = pd.read_csv('data/loan_approval_dataset.csv')
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

df['education'] = df['education'].map({'Graduate': 1, 'Not Graduate': 0})
df['self_employed'] = df['self_employed'].map({'Yes': 1, 'No': 0})
df['loan_status'] = df['loan_status'].map({'Approved': 1, 'Rejected': 0})

df['total_assets_value'] = df['residential_assets_value'] + df['commercial_assets_value'] + df['luxury_assets_value'] + df['bank_asset_value']
df['debt_to_income_ratio'] = df['loan_amount'] / df['income_annum']

df.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_18316\4100956585.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status,total_assets_value,debt_to_income_ratio
0,1,2,1,0,9600000,29900000,12,778,2400000,17600000,22700000,8000000,1,50700000,3.114583
1,2,0,0,1,4100000,12200000,8,417,2700000,2200000,8800000,3300000,0,17000000,2.975610
2,3,3,1,0,9100000,29700000,20,506,7100000,4500000,33300000,12800000,0,57700000,3.263736
3,4,3,1,0,8200000,30700000,8,467,18200000,3300000,23300000,7900000,0,52700000,3.743902
4,5,5,0,1,9800000,24200000,20,382,12400000,8200000,29400000,5000000,0,55000000,2.469388


In [2]:
df_approved = df[df['loan_status'] == 1].copy()
print('Total approved applicants:', df_approved.shape[0])

Total approved applicants: 2656


In [4]:
from sklearn.model_selection import train_test_split

X_reg = df_approved.drop(columns=['loan_id', 'loan_status', 'loan_amount', 'debt_to_income_ratio'])
y_reg = df_approved['loan_amount']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

In [5]:
from sklearn.preprocessing import StandardScaler

scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

In [6]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [7]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_reg_scaled, y_train_reg)
y_pred_lin = lin_reg.predict(X_test_reg_scaled)

In [8]:
ridge_reg = Ridge(random_state=42)
ridge_reg.fit(X_train_reg_scaled, y_train_reg)
y_pred_ridge = ridge_reg.predict(X_test_reg_scaled)

In [9]:
dt_reg = DecisionTreeRegressor(random_state=42)
dt_reg.fit(X_train_reg_scaled, y_train_reg)
y_pred_dt_reg = dt_reg.predict(X_test_reg_scaled)

In [10]:
rf_reg = RandomForestRegressor(random_state=42)
rf_reg.fit(X_train_reg_scaled, y_train_reg)
y_pred_rf_reg = rf_reg.predict(X_test_reg_scaled)

In [11]:
svr_reg = SVR()
svr_reg.fit(X_train_reg_scaled, y_train_reg)
y_pred_svr = svr_reg.predict(X_test_reg_scaled)

In [12]:
def evaluate_regression(name, y_true, y_pred):
    return {
        'Model': name,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2 Score': r2_score(y_true, y_pred)
    }

In [13]:
reg_results = []
reg_results.append(evaluate_regression('Linear Regression', y_test_reg, y_pred_lin))
reg_results.append(evaluate_regression('Ridge Regression', y_test_reg, y_pred_ridge))
reg_results.append(evaluate_regression('Decision Tree', y_test_reg, y_pred_dt_reg))
reg_results.append(evaluate_regression('Random Forest', y_test_reg, y_pred_rf_reg))
reg_results.append(evaluate_regression('SVR', y_test_reg, y_pred_svr))

reg_results_df = pd.DataFrame(reg_results)
reg_results_df

,Model,MAE,RMSE,R2 Score
0,Linear Regression,2.504399e+06,3.308121e+06,0.872677
1,Ridge Regression,2.504177e+06,3.308104e+06,0.872678
2,Decision Tree,3.409774e+06,4.718775e+06,0.740938
3,Random Forest,2.457395e+06,3.319540e+06,0.871796
4,SVR,7.767900e+06,9.282312e+06,-0.002439


In [14]:
import joblib

joblib.dump(rf_reg, 'random_forest_regressor.pkl')
joblib.dump(scaler_reg, 'scaler_reg.pkl')

print('Regression model saved successfully')

Regression model saved successfully
